# 15 — HOG + Color Dual-Branch MLP
HOG: **34.992**; Color: **439**. HOG dalına RAM dostu sabit rastgele projeksiyon (128 boyut) uygulanır; Color dalı 439 özellik. Ağ Keras Functional API ile iki girişlidir. Original ve Masked ayrı eğitim. Yalnızca train üzerinde scaler fit, val ile early stopping (bu validation aynı zamanda raporlama için kullanıldığından skor seçimi iyimser olabilir); test kullanılmaz.

In [1]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import BASE_DIR
from src.data_loader import load_dataset
FEATURE_DIR = BASE_DIR / "outputs" / "features"
RESULT_DIR = BASE_DIR / "outputs" / "models" / "deep_learning"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset(require_masks=True).reset_index(drop=True)
y = df['label'].to_numpy(dtype=str)
ids = df['image'].to_numpy(dtype=str)
classes = np.unique(y)
idx = np.arange(len(y))
dev_idx, test_idx = train_test_split(idx, test_size=.15, random_state=42, stratify=y)
train_idx, val_idx = train_test_split(dev_idx, test_size=.15/.85, random_state=42, stratify=y[dev_idx])
assert len(set(train_idx)&set(val_idx)) == len(set(train_idx)&set(test_idx)) == len(set(val_idx)&set(test_idx)) == 0
print('Train:',len(train_idx),'Val:',len(val_idx),'Test (kullanılmayacak):',len(test_idx))
# ÖNEMLİ: Aynı lezyondan farklı görüntüler varsa lesion_id ile group split ayrıca uygulanmalı.


Train: 7009 Val: 1503 Test (kullanılmayacak): 1503


In [4]:
paths = {'original': FEATURE_DIR/'X_original_color_300x225.npy', 'masked':FEATURE_DIR/'X_masked_color_300x225.npy'}
ids_file = FEATURE_DIR/'image_ids_color_300x225.npy'
y_file = FEATURE_DIR/'y_labels_color_300x225.npy'
assert np.array_equal(np.load(ids_file,allow_pickle=False).astype(str), ids)
assert np.array_equal(np.load(y_file,allow_pickle=False).astype(str), y)
for p in paths.values():
    x0=np.load(p,mmap_mode='r',allow_pickle=False)
    assert x0.shape == (len(y),439), (p,x0.shape)
print('Color ID/etiket sırası doğrulandı.')


Color ID/etiket sırası doğrulandı.


In [3]:
import tensorflow as tf
from sklearn.random_projection import SparseRandomProjection
from sklearn.preprocessing import StandardScaler, LabelEncoder
from joblib import dump
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
HOG_PATHS={'original':FEATURE_DIR/'X_original_hog_300x225.npy','masked':FEATURE_DIR/'X_masked_hog_300x225.npy'}
for variant,hp in HOG_PATHS.items():
    h=np.load(hp,mmap_mode='r',allow_pickle=False)
    assert h.shape==(len(y),34992),(variant,h.shape)
# HOG ve Color matrislerinin satır sırası aynı görüntü ID'leriyle doğrulanır
assert np.array_equal(np.load(FEATURE_DIR/'image_ids.npy',allow_pickle=True).astype(str), ids), 'HOG ID sırası Color ile eşleşmiyor!'
encoder=LabelEncoder().fit(y);yi=encoder.transform(y)
HOG_DIM=128;BATCH=128;EPOCHS=80


In [5]:
def store_result(model_name,variant,pred,train_pred,best_params,cv_score=None,seconds=None):
    prefix=RESULT_DIR/f'{model_name}_{variant}'
    r={'Model':model_name,'Variant':variant,'CV_Macro_F1':cv_score,
       'Train_Macro_F1':float(f1_score(y[train_idx],train_pred,average='macro',zero_division=0)),
       'Val_Accuracy':float(accuracy_score(y[val_idx],pred)),
       'Val_Macro_F1':float(f1_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Precision':float(precision_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Recall':float(recall_score(y[val_idx],pred,average='macro',zero_division=0)),
       'MEL_Recall':float(recall_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'MEL_Precision':float(precision_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'Predicted_Classes':int(len(np.unique(pred))),'Best_Params':json.dumps(best_params,ensure_ascii=False,default=str),
       'Search_Seconds':seconds,'Train_N':len(train_idx),'Val_N':len(val_idx),'Test_N':len(test_idx)}
    pd.DataFrame({'image_id':ids[val_idx], 'actual':y[val_idx], 'predicted':pred}).to_csv(str(prefix)+'_val_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y[val_idx],pred,labels=classes),index=classes,columns=classes).to_csv(str(prefix)+'_confusion_matrix.csv')
    print(classification_report(y[val_idx],pred,labels=classes,zero_division=0))
    return r


In [6]:
def hog_project(hog_matrix, indices, projector, batch_size=128):
    result=np.empty((len(indices),HOG_DIM),dtype=np.float32)
    for start in range(0,len(indices),batch_size):
        ids_batch=indices[start:start+batch_size]
        result[start:start+len(ids_batch)]=projector.transform(np.asarray(hog_matrix[ids_batch],dtype=np.float32))
    return result

def build_model():
    h_in=layers.Input((HOG_DIM,),name='hog_input')
    c_in=layers.Input((439,),name='color_input')
    h=layers.Dense(64,activation='relu')(h_in);h=layers.Dropout(.35)(h)
    c=layers.Dense(64,activation='relu')(c_in);c=layers.Dropout(.35)(c)
    z=layers.Concatenate()([h,c]);z=layers.Dense(64,activation='relu')(z);z=layers.Dropout(.4)(z)
    out=layers.Dense(len(classes),activation='softmax')(z)
    model=Model([h_in,c_in],out);model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model

rows=[]
for variant in ('original','masked'):
    tf.keras.backend.clear_session();tf.keras.utils.set_random_seed(42);started=time.time()
    hog=np.load(HOG_PATHS[variant],mmap_mode='r',allow_pickle=False)
    col=np.load(paths[variant],mmap_mode='r',allow_pickle=False)
    projector=SparseRandomProjection(n_components=HOG_DIM,random_state=42)
    # Yalnız özellik boyutuna bağlı sabit rastgele projeksiyon; etiket kullanılmaz.
    projector.fit(np.zeros((1,hog.shape[1]),dtype=np.float32))
    ht=hog_project(hog,train_idx,projector);hv=hog_project(hog,val_idx,projector)
    ct=np.asarray(col[train_idx],dtype=np.float32);cv=np.asarray(col[val_idx],dtype=np.float32)
    hs=StandardScaler().fit(ht);cs=StandardScaler().fit(ct)
    ht=hs.transform(ht);hv=hs.transform(hv);ct=cs.transform(ct);cv=cs.transform(cv)
    model=build_model();prefix=RESULT_DIR/f'Dual_Branch_MLP_{variant}'
    callbacks=[EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True),
       ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=4,min_lr=1e-6),
       ModelCheckpoint(str(prefix)+'_best.keras',monitor='val_loss',save_best_only=True)]
    history=model.fit({'hog_input':ht,'color_input':ct},yi[train_idx],
       validation_data=({'hog_input':hv,'color_input':cv},yi[val_idx]),epochs=EPOCHS,
       batch_size=BATCH,callbacks=callbacks,verbose=1)
    pred=encoder.inverse_transform(model.predict({'hog_input':hv,'color_input':cv},verbose=0).argmax(1))
    tr=encoder.inverse_transform(model.predict({'hog_input':ht,'color_input':ct},verbose=0).argmax(1))
    dump({'projection':projector,'hog_scaler':hs,'color_scaler':cs,'encoder':encoder},str(prefix)+'_preprocess.joblib')
    pd.DataFrame(history.history).to_csv(str(prefix)+'_history.csv',index=False)
    rows.append(store_result('Dual_Branch_MLP',variant,pred,tr,{'hog_dim':HOG_DIM,'epochs':len(history.history['loss'])},None,round(time.time()-started,1)))
summary=pd.DataFrame(rows);summary.to_csv(RESULT_DIR/'Dual_Branch_MLP_summary.csv',index=False);display(summary)



Epoch 1/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.5969 - loss: 1.2685 - val_accuracy: 0.6693 - val_loss: 0.9387 - learning_rate: 0.0010
Epoch 2/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6550 - loss: 1.0031 - val_accuracy: 0.6707 - val_loss: 0.8912 - learning_rate: 0.0010
Epoch 3/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6738 - loss: 0.9204 - val_accuracy: 0.6780 - val_loss: 0.8563 - learning_rate: 0.0010
Epoch 4/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6865 - loss: 0.8826 - val_accuracy: 0.6813 - val_loss: 0.8359 - learning_rate: 0.0010
Epoch 5/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6877 - loss: 0.8577 - val_accuracy: 0.6893 - val_loss: 0.8168 - learning_rate: 0.0010
Epoch 6/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6977 - loss: 0.8197 - val_accuracy: 0.7046 - val_loss: 0.8064 - learning_rate: 0.0010
Epoch 7/80
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7085 - loss: 0.7869 - val_accuracy:

,Model,Variant,CV_Macro_F1,Train_Macro_F1,Val_Accuracy,Val_Macro_F1,Val_Macro_Precision,Val_Macro_Recall,MEL_Recall,MEL_Precision,Predicted_Classes,Best_Params,Search_Seconds,Train_N,Val_N,Test_N
0,Dual_Branch_MLP,original,None,0.596865,0.723886,0.351036,0.471014,0.327256,0.209581,0.486111,6,"{""hog_dim"": 128, ""epochs"": 28}",19.2,7009,1503,1503
1,Dual_Branch_MLP,masked,None,0.535016,0.720559,0.401865,0.453135,0.390700,0.161677,0.529412,6,"{""hog_dim"": 128, ""epochs"": 23}",18.7,7009,1503,1503


In [ ]:
import winsound

winsound.Beep(300, 3000)
print("Ses testi tamamlandı.")